In [1]:
from importlib import reload

In [2]:
import numpy as np
import tensorflow as tf
from InpaintingModel import InpaintingModel

2025-09-05 15:09:17.920962: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757077757.941487  904960 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757077757.947934  904960 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1757077757.964726  904960 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1757077757.964753  904960 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1757077757.964755  904960 computation_placer.cc:177] computation placer alr

In [8]:
# Dummy data
batch_size = 2
field_size = 16
x_dummy = np.random.rand(batch_size, field_size, field_size, field_size, 2).astype(np.float32)
y_dummy = np.random.rand(batch_size, field_size, field_size, field_size, 1).astype(np.float32)
mask_dummy = np.random.randint(0,2,size=(batch_size, field_size, field_size, field_size,1)).astype(np.float32)

In [9]:
# Build model
base_model = InpaintingModel().prepare_model(input_size=(field_size,field_size,field_size,2))

# Minimal Masked model
class MaskedInpaintingModel(tf.keras.Model):
    def train_step(self, data):
        x, y_true, mask = data
        with tf.GradientTape() as tape:
            y_pred = self(x, training=True)
            missing_mask = 1.0 - mask
            loss = tf.reduce_sum((y_true - y_pred)**2 * missing_mask) / (tf.reduce_sum(missing_mask)+1e-8)
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        return {"loss": loss}

In [10]:
masked_model = MaskedInpaintingModel(inputs=base_model.input, outputs=base_model.output)
masked_model.compile(optimizer=tf.keras.optimizers.Adam())

# Run one training step
masked_model.train_step((x_dummy, y_dummy, mask_dummy))
print("Test step ran successfully")


/farmdisk1/emaragliano/miniconda3/envs/tf/lib/python3.9/site-packages/keras/src/models/functional.py:238: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['keras_tensor_18']
Received: inputs=Tensor(shape=(2, 16, 16, 16, 2))
  warnings.warn(msg)


Test step ran successfully


In [14]:
import os
import shutil

# Cartella sorgente
src_dir = "/home/emaragliano/Work/Projects/myfarm-disk/QuijoteSimulations/QuijoteFields_RealSpace/groups_002"

# Cartella destinazione
dst_dir = "/home/emaragliano/Work/Projects/myfarm-disk/AE_storage_Paper2/AE_model_fill_holes/true_data"

# Crea la cartella di destinazione se non esiste
os.makedirs(dst_dir, exist_ok=True)

# Ciclo sui numeri da 000 a 099
for i in range(100):
    folder_name = f"{i:03d}"  # 000, 001, ..., 099
    src_file = os.path.join(src_dir, folder_name, f"halo_over_density_real_space_128_CIC_{str(i).zfill(3)}_group_tab_002.npy")
    
    if os.path.exists(src_file):
        shutil.copy(src_file, dst_dir)
        print(f"Copiato {src_file} -> {dst_dir}")
    else:
        print(f"File non trovato: {src_file}")


Copiato /home/emaragliano/Work/Projects/myfarm-disk/QuijoteSimulations/QuijoteFields_RealSpace/groups_002/000/halo_over_density_real_space_128_CIC_000_group_tab_002.npy -> /home/emaragliano/Work/Projects/myfarm-disk/AE_storage_Paper2/AE_model_fill_holes/true_data
Copiato /home/emaragliano/Work/Projects/myfarm-disk/QuijoteSimulations/QuijoteFields_RealSpace/groups_002/001/halo_over_density_real_space_128_CIC_001_group_tab_002.npy -> /home/emaragliano/Work/Projects/myfarm-disk/AE_storage_Paper2/AE_model_fill_holes/true_data
Copiato /home/emaragliano/Work/Projects/myfarm-disk/QuijoteSimulations/QuijoteFields_RealSpace/groups_002/002/halo_over_density_real_space_128_CIC_002_group_tab_002.npy -> /home/emaragliano/Work/Projects/myfarm-disk/AE_storage_Paper2/AE_model_fill_holes/true_data
Copiato /home/emaragliano/Work/Projects/myfarm-disk/QuijoteSimulations/QuijoteFields_RealSpace/groups_002/003/halo_over_density_real_space_128_CIC_003_group_tab_002.npy -> /home/emaragliano/Work/Projects/myfa

In [4]:
ones_grid = np.random.choice([0, 1], size=(128, 128, 128), p=[0.03, 0.97]).astype(np.float32)
np.save("mask_data/ones_grid.npy", ones_grid)